# Predictive modelling and regression analysis

This notebook applies multiple linear regression and binary logistic regression to the Adult Income dataset (1994 US Census).

| Part  | Dependent variable (Y) |
|---|---|
| Multiple linear regression |  `hours_per_week` (continuous) |
| Binary logistic regression |  `income` (>50K = 1, ≤50K = 0) |

The models are fitted on the training file (`adult.data`) and evaluated on the separate test file (`adult.test`).

The default significance level is α = 0.05.

## Getting ready

Import libraries.

In [ ]:
# Import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from scipy import stats

ALPHA = 0.05
RANDOM_STATE = 42

print("Libraries loaded.")

## Load the data

The data was cleaned and encoded in the descriptive statistics notebook and saved to `processed_data`. Both files contain the 41 encoded predictors plus the target `income` (1 = >50K, 0 = ≤50K).

In [ ]:
train = pd.read_csv("C:/Users/gabij/OneDrive/Desktop/processed_data/train_encoded.csv")
test = pd.read_csv("C:/Users/gabij/OneDrive/Desktop/processed_data/test_encoded.csv")

# Safety checks: remove an index column (if saved with the index) and any leftover helper column
extra_columns = [c for c in train.columns if c.startswith("Unnamed") or c == "education_te"]
train = train.drop(columns=extra_columns)
test = test.drop(columns=extra_columns, errors="ignore")

print("Training shape:", train.shape)   # expected about (30162, 42): 41 predictors + income
print("Test shape:    ", test.shape)    # expected about (15060, 42)
print("Missing values (train):", train.isna().sum().sum())
print("Missing values (test): ", test.isna().sum().sum())
print("Income values:", sorted(train["income"].unique().tolist()))

display(train.head())

#### Fix the test labels

In `test_encoded.csv` every `income` value is 0. The raw `adult.test` file writes the labels as `<=50K.` and `>50K.` (with a trailing period), so the `>50K` mapping in the descriptive statistics notebook never matched. The correct labels are re-read from the raw `adult.test` file here. No rows were removed during encoding (16,281 rows in both files), so the rows line up one to one. The `age` column is compared to confirm this.

In [ ]:
import os

# Path to the raw UCI test file. If it isn't found, the file is downloaded from the UCI repository.
RAW_TEST_PATH = "C:/Users/gabij/OneDrive/Desktop/adult.test"
RAW_TEST_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

raw_columns = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]

if test["income"].nunique() < 2:
    raw_source = RAW_TEST_PATH if os.path.exists(RAW_TEST_PATH) else RAW_TEST_URL
    raw_test = pd.read_csv(
        raw_source,
        header=None,
        names=raw_columns,
        skiprows=1,              # the first line of adult.test is "|1x3 Cross validator"
        skipinitialspace=True
    )

    # Remove the trailing period and spaces: ">50K." -> ">50K"
    raw_income = raw_test["income"].astype(str).str.strip().str.rstrip(".")

    assert len(raw_test) == len(test), "Row counts differ. The rows cannot be matched one to one."
    assert (raw_test["age"].values == test["age"].values).all(), "Rows are not in the same order."

    test["income"] = (raw_income == ">50K").astype(int).values


print(test["income"].value_counts().rename({0: "<=50K (0)", 1: ">50K (1)"}))

# Multiple linear regression

Variables:
- Dependent variable (Y): `hours_per_week`
- Independent variables (X):
  - `age` (years)
  - `education_num` (years of education, 1–16)
  - `gender_male` (1 = Male, 0 = Female)

#### Prepare the data and display summary statistics

In [ ]:
linear_predictors = ["age", "education_num", "gender_male"]
linear_target = "hours_per_week"

def prepare_linear_data(data):
    # In the encoded data, gender is already 1 = Male, 0 = Female
    prepared = data[["age", "education_num", "gender", "hours_per_week"]].copy()
    prepared = prepared.rename(columns={"gender": "gender_male"})
    return prepared[linear_predictors + [linear_target]]

linear_train = prepare_linear_data(train)
linear_test = prepare_linear_data(test)

print("Training data shape:", linear_train.shape)
print("Test data shape:    ", linear_test.shape)

display(linear_train.head())

print("Summary statistics (training data)")
display(linear_train.describe().T.round(3))

#### Correlation matrix and significance testing

Pearson correlation between every pair of variables. For each pair:

- H0: ρ = 0 (no linear correlation)
- H1: ρ ≠ 0

In [ ]:
correlation_matrix = linear_train.corr(method="pearson")

display(correlation_matrix.round(4))

plt.figure(figsize=(7, 5))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix — Multiple Linear Regression")
plt.show()


# Significance tests for all correlations

correlation_results = []
columns = linear_train.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):

        var1 = columns[i]
        var2 = columns[j]

        r, p_value = stats.pearsonr(linear_train[var1], linear_train[var2])

        correlation_results.append({
            "Variable 1": var1,
            "Variable 2": var2,
            "Correlation (r)": round(r, 4),
            "p-value": p_value,
            "Significant": "Yes" if p_value < ALPHA else "No"
        })

correlation_significance = pd.DataFrame(correlation_results)

print("H0: ρ = 0    H1: ρ ≠ 0")
display(correlation_significance)

#### Scatter plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(linear_train["age"], linear_train["hours_per_week"], alpha=0.1, s=8)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Hours per week")
axes[0].set_title("Age vs. Hours per week")

axes[1].scatter(linear_train["education_num"], linear_train["hours_per_week"], alpha=0.1, s=8)
axes[1].set_xlabel("Years of education")
axes[1].set_ylabel("Hours per week")
axes[1].set_title("Education vs. Hours per week")

sns.boxplot(
    x=linear_train["gender_male"].map({0: "Female", 1: "Male"}),
    y=linear_train["hours_per_week"],
    ax=axes[2]
)
axes[2].set_xlabel("Gender")
axes[2].set_ylabel("Hours per week")
axes[2].set_title("Gender vs. Hours per week")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Model fitting: coefficients and regression equation

The model is fitted on the training data. The same coefficients are estimated with scikit-learn (`LinearRegression`) and with statsmodels (`OLS`), which also gives the significance tests.

In [ ]:
X_train_linear = linear_train[linear_predictors]
y_train_linear = linear_train[linear_target]

X_test_linear = linear_test[linear_predictors]
y_test_linear = linear_test[linear_target]

print("Training observations:", len(X_train_linear))
print("Testing observations: ", len(X_test_linear))

# Fit multiple regression model (scikit-learn)

linear_model = LinearRegression()
linear_model.fit(X_train_linear, y_train_linear)

linear_intercept = linear_model.intercept_
linear_coefficients = pd.Series(linear_model.coef_, index=linear_predictors)

print("\nMultiple Linear Regression model")
print("Coefficients:")
print(f"Intercept (b0) = {linear_intercept:.6f}")
for variable, coefficient in linear_coefficients.items():
    print(f"{variable} coefficient = {coefficient:.6f}")

print("\nEstimated regression equation:")
equation = f"hours_per_week = {linear_intercept:.4f}"
for variable, coefficient in linear_coefficients.items():
    sign = "+" if coefficient >= 0 else "-"
    equation += f" {sign} {abs(coefficient):.4f} × {variable}"
print(equation)


# ============================================================
# Using Ordinary Least Squares (OLS) method
# ============================================================

X_train_linear_const = sm.add_constant(X_train_linear)

ols_linear_model = sm.OLS(y_train_linear, X_train_linear_const).fit()

print(ols_linear_model.summary())

#### Testing for significance (*t*-tests)

For each coefficient:
- H0: bᵢ = 0 (the variable is not a significant predictor)
- H1: bᵢ ≠ 0

In [ ]:
print("t-tests for individual regression coefficients")

t_test_results = []

for variable in linear_predictors:

    t_stat = ols_linear_model.tvalues[variable]
    p_value = ols_linear_model.pvalues[variable]

    print(f"\n{variable}")
    print(f"H0: b({variable}) = 0")
    print(f"H1: b({variable}) ≠ 0")
    print(f"t-statistic = {t_stat:.4f}")
    print(f"p-value = {p_value:.6f}")

    if p_value < ALPHA:
        print(f"p-value < {ALPHA}: Reject H0. {variable} is a statistically significant predictor.")
    else:
        print(f"p-value >= {ALPHA}: Do not reject H0. There is insufficient evidence that {variable} is a significant predictor.")

    t_test_results.append({
        "Variable": variable,
        "Coefficient": ols_linear_model.params[variable],
        "t-statistic": t_stat,
        "p-value": p_value,
        "Significant": "Yes" if p_value < ALPHA else "No"
    })

display(pd.DataFrame(t_test_results).round(6))

#### Testing for significance (*F*-test)

In [ ]:
print("F-test for the overall regression model")
print("H0: b1 = b2 = b3 = 0")
print("H1: At least one b coefficient is not equal to 0")

f_stat_linear = ols_linear_model.fvalue
f_p_value_linear = ols_linear_model.f_pvalue

print(f"F-statistic = {f_stat_linear:.4f}")
print(f"p-value = {f_p_value_linear:.6f}")

if f_p_value_linear < ALPHA:
    print(f"p-value < {ALPHA}: Reject H0. The regression model is statistically significant.")
else:
    print(f"p-value >= {ALPHA}: Do not reject H0. There is insufficient evidence that the regression model is statistically significant.")

#### Test set predictions

In [ ]:
y_pred_linear = linear_model.predict(X_test_linear)

linear_predictions = X_test_linear.copy()

linear_predictions["Actual"] = y_test_linear.values
linear_predictions["Predicted"] = y_pred_linear
linear_predictions["Residual"] = linear_predictions["Actual"] - linear_predictions["Predicted"]

display(linear_predictions.head(15).round(3))

#### Actual vs. predicted plot

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    linear_predictions["Actual"],
    linear_predictions["Predicted"],
    alpha=0.15,
    s=10
)

min_value = min(linear_predictions["Actual"].min(), linear_predictions["Predicted"].min())
max_value = max(linear_predictions["Actual"].max(), linear_predictions["Predicted"].max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    color="black",
    label="Perfect prediction"
)

plt.xlabel("Actual hours per week")
plt.ylabel("Predicted hours per week")
plt.title("Actual vs. Predicted Values — Multiple Linear Regression")

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

#### Predictive performance (model evaluation)

In [ ]:
def calculate_mape(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE).
    Observations with actual values equal to zero
    are excluded to avoid division by zero.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    nonzero_mask = y_true != 0

    if nonzero_mask.sum() == 0:
        return np.nan

    return np.mean(
        np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])
    ) * 100


mae_linear = mean_absolute_error(y_test_linear, y_pred_linear)
mse_linear = mean_squared_error(y_test_linear, y_pred_linear)
rmse_linear = np.sqrt(mse_linear)
mape_linear = calculate_mape(y_test_linear, y_pred_linear)
r_squared_linear = r2_score(y_test_linear, y_pred_linear)

linear_metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "MAPE (%)", "R-squared"],
    "Value": [mae_linear, mse_linear, rmse_linear, mape_linear, r_squared_linear]
})

display(linear_metrics.round(4))

print(f"R-squared on training data (from OLS) = {ols_linear_model.rsquared:.4f}")

#### Predictions for new data points

In [ ]:
new_people_linear = pd.DataFrame({
    "age":           [25, 40, 40, 55],
    "education_num": [9, 13, 13, 16],
    "gender_male":   [0, 0, 1, 1]
})

new_people_linear["Predicted hours_per_week"] = linear_model.predict(
    new_people_linear[linear_predictors]
)

print("Predicted weekly working hours for new people")
print("(education_num: 9 = HS-grad, 13 = Bachelors, 16 = Doctorate; gender_male: 1 = Male, 0 = Female)")
display(new_people_linear.round(2))

# Binary logistic regression

Variables:
- Dependent variable (Y): `income` (1 = >50K, 0 = ≤50K)
- Independent variables (X): all characteristics after encoding (41 numeric columns)

#### Prepare the data

The data is already encoded (done in the descriptive statistics notebook):
- `gender` → 1 = Male, 0 = Female
- `native_country` → `is_us` (1 = United States, 0 = other)
- `workclass`, `marital_status`, `occupation`, `relationship`, `race` → one-hot encoding (the first category of each is the reference group: Federal-gov, Divorced, Adm-clerical, Husband, Amer-Indian-Eskimo)
- `education` was dropped because `education_num` contains the same information as a number
- `fnlwgt` was dropped because it is a census sampling weight, not a characteristic of the person

In [ ]:
X_train_log = train.drop(columns="income")
y_train_log = train["income"].astype(int)

# Make sure test has exactly the same columns, in the same order, as train
X_test_log = test.drop(columns="income").reindex(columns=X_train_log.columns, fill_value=0)
y_test_log = test["income"].astype(int)

assert y_test_log.nunique() == 2, "The test set has only one income class. Run the label fix cell above."

logistic_predictors = X_train_log.columns.tolist()

print("X_train shape:", X_train_log.shape)
print("X_test shape: ", X_test_log.shape)
print("Non-numeric columns:", X_train_log.select_dtypes(exclude="number").columns.tolist())

display(X_train_log.head())

#### Target variable distribution

In [ ]:
target_summary = pd.DataFrame({
    "Train count": y_train_log.value_counts().sort_index(),
    "Train (%)": y_train_log.value_counts(normalize=True).sort_index().mul(100),
    "Test count": y_test_log.value_counts().sort_index(),
    "Test (%)": y_test_log.value_counts(normalize=True).sort_index().mul(100)
})

target_summary.index = ["<=50K (0)", ">50K (1)"]

display(target_summary.round(2))

#### Fit the logistic regression model

The predictors are on very different scales (for example, `capital_gain` goes up to 99,999 while the one-hot columns are 0 or 1). They are standardised with `StandardScaler` inside a pipeline so the model fits reliably. The coefficients are then converted back to the original units, so the equation can be used with ordinary values.

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

logistic_model.fit(X_train_log, y_train_log)

# Predict probabilities
# Column 0 = probability of class 0 (<=50K)
# Column 1 = probability of class 1 (>50K)

y_prob_log = logistic_model.predict_proba(X_test_log)[:, 1]

# Predict classes using the default threshold of 0.50

classification_threshold = 0.50

y_pred_log = (y_prob_log >= classification_threshold).astype(int)

print("Logistic regression model fitted successfully.")
print(f"Classification threshold = {classification_threshold}")

#### Coefficients and logistic regression equation


In [ ]:
scaler = logistic_model.named_steps["scaler"]
logistic = logistic_model.named_steps["logistic"]

standardised_coefficients = logistic.coef_[0]

# Convert coefficients back to the original units of each variable
original_coefficients = standardised_coefficients / scaler.scale_
original_intercept = logistic.intercept_[0] - np.sum(standardised_coefficients * scaler.mean_ / scaler.scale_)

logistic_coefficients_table = pd.DataFrame({
    "Variable": logistic_predictors,
    "Coefficient": original_coefficients,
    "Odds_Ratio": np.exp(original_coefficients),
    "Standardised_Coefficient": standardised_coefficients
}).sort_values("Standardised_Coefficient", key=abs, ascending=False)

display(logistic_coefficients_table.round(6).reset_index(drop=True))

print(f"Intercept = {original_intercept:.6f}")

print("\nLogistic regression equation (log-odds scale):")

equation = f"logit(p) = {original_intercept:.6f}"

for variable, coefficient in zip(logistic_predictors, original_coefficients):
    sign = "+" if coefficient >= 0 else "-"
    equation += f"\n           {sign} {abs(coefficient):.6f} × {variable}"

print(equation)

print("\nwhere p = P(income > 50K) = 1 / (1 + e^(-logit(p)))")

#### Predicted probabilities

Predicted probability of earning >50K ("Yes") for each person in the test set, and the predicted class using the 0.50 threshold.

In [ ]:
logistic_predictions = X_test_log[["age", "education_num", "hours_per_week", "gender"]].copy()

logistic_predictions["Actual_Income"] = y_test_log.map({0: "No (<=50K)", 1: "Yes (>50K)"}).values
logistic_predictions["Probability_No"] = 1 - y_prob_log
logistic_predictions["Probability_Yes"] = y_prob_log
logistic_predictions["Predicted_Income"] = pd.Series(y_pred_log).map({0: "No (<=50K)", 1: "Yes (>50K)"}).values

display(logistic_predictions.head(15).round(4))

#### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test_log, y_pred_log, labels=[0, 1])

tn, fp, fn, tp = cm.ravel()

print("Confusion matrix:")
print(cm)
print(f"\nTrue negatives  (correctly predicted <=50K): {tn}")
print(f"False positives (predicted >50K, actually <=50K): {fp}")
print(f"False negatives (predicted <=50K, actually >50K): {fn}")
print(f"True positives  (correctly predicted >50K): {tp}")

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No (<=50K)", "Yes (>50K)"],
    yticklabels=["No (<=50K)", "Yes (>50K)"]
)

plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("Confusion Matrix — Binary Logistic Regression")
plt.show()

#### Classification metrics (requirement 4)

In [ ]:
accuracy = accuracy_score(y_test_log, y_pred_log)
precision = precision_score(y_test_log, y_pred_log, zero_division=0)
recall = recall_score(y_test_log, y_pred_log, zero_division=0)
f1 = f1_score(y_test_log, y_pred_log, zero_division=0)
auc = roc_auc_score(y_test_log, y_prob_log)

classification_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, auc]
})

display(classification_metrics.round(4))

print(f"Baseline accuracy (always predicting <=50K) = {1 - y_test_log.mean():.4f}")

**How to read the metrics:**
- **Accuracy:** share of all people classified correctly. Compare it with the baseline: about 75% of people earn ≤50K, so a model that always says "No" would already be about 75% accurate.
- **Precision:** of the people the model predicts earn >50K, the share who actually do.
- **Recall:** of the people who actually earn >50K, the share the model finds.
- **F1-score:** balance between precision and recall.
- **ROC-AUC:** how well the model ranks high earners above low earners across all thresholds (0.5 = random guessing, 1.0 = perfect).

#### Classification report

In [ ]:
print("Classification report:")

print(
    classification_report(
        y_test_log,
        y_pred_log,
        target_names=["No (<=50K)", "Yes (>50K)"],
        zero_division=0
    )
)

#### ROC curve and AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_log, y_prob_log)

roc_auc = roc_auc_score(y_test_log, y_prob_log)

plt.figure(figsize=(8, 6))

plt.plot(fpr, tpr, linewidth=2, label=f"Logistic Regression (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5, label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Binary Logistic Regression")

plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"ROC-AUC = {roc_auc:.4f}")

#### User-defined prediction (requirement 5)

Enter a new person's characteristics in readable form. The helper function turns them into the same 41 encoded columns as the training data (a reference category such as `Husband` or `Federal-gov` simply means all of that variable's columns are 0). The model then predicts the probability of "No" (≤50K) and "Yes" (>50K).

In [ ]:
# Replace with the desired values
new_person = {
    "age": 40,
    "education_num": 13,                    # 9 = HS-grad, 13 = Bachelors, 16 = Doctorate
    "gender": "Male",
    "capital_gain": 0,
    "capital_loss": 0,
    "hours_per_week": 45,
    "native_country": "United-States",
    "workclass": "Private",
    "marital_status": "Married-civ-spouse",
    "occupation": "Exec-managerial",
    "relationship": "Husband",
    "race": "White"
}


def encode_new_person(person, columns):
    """Turn readable values into one row with the same encoded columns as the training data."""
    row = pd.DataFrame(0, index=[0], columns=columns)

    for variable in ["age", "education_num", "capital_gain", "capital_loss", "hours_per_week"]:
        row[variable] = person[variable]

    row["gender"] = 1 if person["gender"] == "Male" else 0
    row["is_us"] = 1 if person["native_country"] == "United-States" else 0

    for variable in ["workclass", "marital_status", "occupation", "relationship", "race"]:
        column = f"{variable}_{person[variable]}"
        if column in columns:
            row[column] = 1
        # otherwise it is the reference category, so all columns stay 0

    return row


new_person_encoded = encode_new_person(new_person, logistic_predictors)

probabilities = logistic_model.predict_proba(new_person_encoded)[0]

prob_no = probabilities[0]
prob_yes = probabilities[1]

prediction = "Yes (>50K)" if prob_yes >= classification_threshold else "No (<=50K)"

print("=" * 50)
print("LOGISTIC REGRESSION PREDICTION")
print("=" * 50)

print("\nInput values:")
display(pd.DataFrame([new_person]))

print(f"\nProbability of No  (<=50K): {prob_no:.4f} ({prob_no * 100:.2f}%)")
print(f"Probability of Yes (>50K):  {prob_yes:.4f} ({prob_yes * 100:.2f}%)")

print(f"\nPredicted income: {prediction}")